# 第15章　随机变量与分布 —— 用采样代替公式

> **本章目标**：理解随机变量的本质——值由概率决定的变量。掌握四种核心分布（离散/正态/均匀/伯努利）、极大似然估计（MLE）的直觉，以及四种采样策略（greedy/temperature/top-k/top-p）。
> **前置知识**：第 1 章（Python/NumPy/matplotlib）、第 3 章（标量与向量）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf
print("✅ 环境就绪 — NumPy + Matplotlib")

---
## 15.1　离散随机变量 —— 掷骰子与概率质量函数

📐 **定义　离散随机变量（Discrete Random Variable）**：取值集合有限或可数，每个取值 $x_i$ 对应概率 $p_i = P(X=x_i)$，满足 $\sum p_i = 1$ 且 $p_i \ge 0$。

概率质量函数（PMF）描述每个取值的概率。最经典的例子是公平骰子：每个面的概率都是 $1/6$。

🔗 **AI 连接**：分类任务的 softmax 输出就是一个离散概率分布（PMF）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, n in enumerate([30, 300, 3000]):
    rolls = np.random.randint(1, 7, size=n)
    full_counts = np.array([np.sum(rolls == k) for k in range(1, 7)])
    probs = full_counts / n

    axes[i].bar(range(1, 7), probs, color='steelblue', edgecolor='white')
    axes[i].axhline(y=1/6, color='red', linestyle='--', linewidth=2, label='理论概率 1/6')
    axes[i].set_title(f'n={n} 次'); axes[i].set_xlabel('点数')
    axes[i].set_ylabel('频率'); axes[i].set_ylim(0, 0.4)
    axes[i].legend(fontsize=8)

plt.suptitle('掷骰子频率随试验次数收敛到 1/6', fontsize=13)
plt.tight_layout(); plt.show()
print("收敛规律：n 越大，柱高越接近红线（大数定律——见第 18 章）")

---
## 15.2　连续随机变量 —— 正态分布与概率密度函数

📐 **定义　连续随机变量（Continuous Random Variable）**：取值于连续区间，由 PDF $f(x)$ 描述，$P(a \le X \le b) = \int_a^b f(x)dx$ = 曲线下面积。

正态分布由两个参数决定：$\mu$（均值/中心）和 $\sigma$（标准差/宽度）。**68-95-99.7 规则**：约 68% 的数据在 $\mu \pm 1\sigma$，约 95% 在 $\mu \pm 2\sigma$，约 99.7% 在 $\mu \pm 3\sigma$。

🔗 **AI 连接**：正态分布是 AI 训练的"默认噪声模型"——权重初始化、BatchNorm、VAE 都依赖它。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf

def normal_pdf(x, mu=0, sigma=1):
    z = (x - mu) / sigma
    return np.exp(-0.5 * z**2) / (sigma * np.sqrt(2 * np.pi))

def normal_cdf(x):
    return 0.5 * (1 + erf(x / np.sqrt(2)))

np.random.seed(42)
mu, sigma = 0.0, 1.0
samples = np.random.randn(10000) * sigma + mu

plt.figure(figsize=(10, 5))
plt.hist(samples, bins=50, density=True, alpha=0.6, color='steelblue',
         edgecolor='white', label='采样直方图')
x = np.linspace(-4, 4, 200)
pdf = normal_pdf(x, mu, sigma)
plt.plot(x, pdf, 'r-', linewidth=2.5, label=f'N({mu}, {sigma}²) 理论 PDF')

for k, color in zip([1, 2, 3], ['orange', 'green', 'purple']):
    plt.axvspan(mu - k*sigma, mu + k*sigma, alpha=0.08, color=color,
                label=f'±{k}σ ({normal_cdf(k)-normal_cdf(-k):.1%})')

plt.xlabel('x'); plt.ylabel('概率密度'); plt.title('标准正态分布 N(0, 1)')
plt.legend(fontsize=8); plt.show()

for k in [1, 2, 3]:
    in_range = np.mean(np.abs(samples) < k)
    expected = normal_cdf(k) - normal_cdf(-k)
    print(f"±{k}σ 内: 采样 {in_range:.3f}  |  理论 {expected:.3f}")

---
## 15.3　均匀分布 —— 等可能的连续版本

📐 **定义　均匀分布（Uniform Distribution）**：在 $[a, b]$ 上 PDF 为常数 $f(x)=1/(b-a)$，区间外为 0。

在机器学习中最经典的用途是**权重初始化**——训练开始时对参数一无所知，用均匀分布随机赋初值。PyTorch 的 `nn.Linear` 默认用 `kaiming_uniform_`。

🔗 **AI 连接**：权重初始化（第 25 章）直接依赖均匀分布和正态分布。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
fan_in, fan_out = 784, 256
limit = np.sqrt(6 / (fan_in + fan_out))
weights = np.random.uniform(-limit, limit, size=(fan_in, fan_out))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(weights.flatten(), bins=50, density=True, color='steelblue', edgecolor='white')
axes[0].axhline(y=1/(2*limit), color='red', linestyle='--', linewidth=2,
                label=f'理论 PDF = 1/(2×{limit:.4f})')
axes[0].set_xlabel('w'); axes[0].set_ylabel('概率密度')
axes[0].set_title(f'Xavier Uniform 初始化\nfan_in={fan_in}, fan_out={fan_out}')
axes[0].legend()

theoretical_var = (2*limit)**2 / 12
axes[1].bar(['理论方差', '实际方差'], [theoretical_var, weights.var()],
            color=['lightgray', 'steelblue'], edgecolor='white')
axes[1].set_ylabel('方差')
axes[1].set_title(f'方差对比 (理论 ≈ {theoretical_var:.6f})')
plt.tight_layout(); plt.show()
print(f"权重范围: [{weights.min():.4f}, {weights.max():.4f}]")
print(f"权重均值: {weights.mean():.6f} (理想=0)")
print(f"权重标准差: {weights.std():.6f}")

---
## 15.4　伯努利分布 —— 二选一的数学表达

📐 **定义　伯努利分布（Bernoulli Distribution）**：$P(X=1)=p$，$P(X=0)=1-p$。只有 0/1 两种结果的离散分布。

应用：二分类输出、Dropout 随机丢弃神经元、点击率预估、A/B 测试。$n$ 次独立伯努利之和服从二项分布 $\text{Binomial}(n, p)$。

🔗 **AI 连接**：Dropout($p$) 就是给每个神经元乘上一个 $\text{Bernoulli}(1-p)$ 随机变量。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
p = 0.3
n_trials = 1000
samples = (np.random.random(n_trials) < p).astype(int)

n_groups = 50
group_successes = samples[:n_groups * 20].reshape(n_groups, 20).sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['失败 (0)', '成功 (1)'],
            [np.mean(samples == 0), np.mean(samples == 1)],
            color=['lightcoral', 'steelblue'], edgecolor='white')
axes[0].set_ylabel('频率')
axes[0].set_title(f'Bernoulli(p={p}) — {n_trials} 次试验')

axes[1].hist(group_successes, bins=range(0, 22), density=True, alpha=0.7,
             color='steelblue', edgecolor='white')
axes[1].set_xlabel('20 次中成功的次数'); axes[1].set_ylabel('频率')
axes[1].set_title(f'20 次伯努利之和 ~ Binomial(20, {p})')
plt.tight_layout(); plt.show()
print(f"样本均值 = {samples.mean():.3f} (理论 p = {p})")
print(f"样本方差 = {samples.var():.3f} (理论 p(1-p) = {p*(1-p):.3f})")

---
## 15.5　极大似然估计（MLE）—— 让观测数据"最可能"出现

📐 **定义　极大似然估计（MLE）**：$\hat{\theta} = \arg\max P(\text{data} \mid \theta)$——选择使观测数据出现概率最大的参数。

对于正态分布 $N(\mu, \sigma^2)$，MLE 给出优雅的闭式解：$\hat{\mu} = \frac{1}{n}\sum x_i$（样本均值），$\hat{\sigma}^2 = \frac{1}{n}\sum(x_i-\hat{\mu})^2$（样本方差）。数据越多，估计越精确。

🔗 **AI 连接**：最大化似然 ⇔ 最小化交叉熵损失（第 19.5 节）。每一个 `nn.CrossEntropyLoss()` 底层都在做 MLE。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
true_mu = 5.0
true_sigma = 2.0
n_total = 200
data = np.random.randn(n_total) * true_sigma + true_mu
cumulative_means = np.cumsum(data) / np.arange(1, n_total + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].axhline(y=true_mu, color='red', linestyle='--', linewidth=2, label=f'真实 μ = {true_mu}')
axes[0].plot(range(1, n_total + 1), cumulative_means, color='steelblue', linewidth=1.5)
axes[0].fill_between(range(1, n_total + 1),
                     true_mu - true_sigma/np.sqrt(np.arange(1, n_total + 1)),
                     true_mu + true_sigma/np.sqrt(np.arange(1, n_total + 1)),
                     alpha=0.15, color='steelblue', label='理论 ±σ/√n')
axes[0].set_xlabel('样本量 n'); axes[0].set_ylabel('μ̂ (估计均值)')
axes[0].set_title('MLE 估计量随样本增大收敛到真值')
axes[0].legend(); axes[0].set_xlim(0, n_total)

for n, color in [(5, 'coral'), (200, 'steelblue')]:
    sub_data = data[:n]
    mu_grid = np.linspace(true_mu - 2, true_mu + 2, 200)
    log_lik = -0.5 * np.sum((sub_data[:, None] - mu_grid[None, :])**2, axis=0) / true_sigma**2
    log_lik = log_lik - log_lik.max()
    axes[1].plot(mu_grid, log_lik, color=color, linewidth=2,
                 label=f'n={n}, μ̂={sub_data.mean():.3f}')
    axes[1].axvline(x=sub_data.mean(), color=color, linestyle='--', alpha=0.5)

axes[1].axvline(x=true_mu, color='red', linestyle='-', alpha=0.6, label=f'真实 μ={true_mu}')
axes[1].set_xlabel('μ'); axes[1].set_ylabel('对数似然 (归一化)')
axes[1].set_title('似然函数随样本量增大而变尖锐')
axes[1].legend()
plt.tight_layout(); plt.show()

print(f"真实 μ = {true_mu}")
print(f"n=5   时 MLE: μ̂ = {data[:5].mean():.3f}")
print(f"n=20  时 MLE: μ̂ = {data[:20].mean():.3f}")
print(f"n=200 时 MLE: μ̂ = {data.mean():.3f}")

---
## 15.6　采样策略 —— 从概率分布到具体选择

有了概率分布后，怎么从中"选一个"？四种策略：

- **Greedy**：永远选 argmax——导致文本循环重复
- **Temperature**：softmax(x/T)，T→0 退化为 greedy，T→∞ 趋近均匀
- **Top-k**：只从概率前 k 高的 token 中选
- **Top-p（核采样）**：从累积概率 ≥ p 的最小集合中选——动态 k，工业标准

🔗 **AI 连接**：第 30.3 节将用 GPT-2 实战这四种策略生成文本。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

logits = np.array([3.0, 2.0, 1.5, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01])

def softmax(x, T=1.0):
    x = np.array(x, dtype=np.float64) / T
    x_max = x.max()
    e_x = np.exp(x - x_max)
    return e_x / e_x.sum()

def sample_from_probs(probs, n_samples=1000):
    return np.random.choice(len(probs), size=n_samples, p=probs)

orig_probs = softmax(logits, T=1.0)

strategies = {}

# Greedy
greedy_idx = np.argmax(orig_probs)
strategies['Greedy'] = np.zeros(10)
strategies['Greedy'][greedy_idx] = 1.0

# Temperature
strategies['Temperature\nT=0.5'] = softmax(logits, T=0.5)
strategies['Temperature\nT=2.0'] = softmax(logits, T=2.0)

# Top-k
top_k = 3
topk_probs = orig_probs.copy()
topk_probs[np.argsort(topk_probs)[:-top_k]] = 0
topk_probs /= topk_probs.sum()
strategies['Top-k\nk=3'] = topk_probs

# Top-p
p = 0.9
sorted_indices = np.argsort(orig_probs)[::-1]
cumsum = np.cumsum(orig_probs[sorted_indices])
cutoff_idx = np.searchsorted(cumsum, p) + 1
topp_probs = orig_probs.copy()
topp_probs[sorted_indices[cutoff_idx:]] = 0
topp_probs /= topp_probs.sum()
strategies['Top-p\np=0.9'] = topp_probs

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
labels = [f'Token_{chr(65+i)}' for i in range(10)]
colors = plt.cm.viridis(np.linspace(0.15, 0.85, 10))

axes[0].bar(labels, orig_probs, color=colors, edgecolor='white')
axes[0].set_title('原始分布 (T=1.0 softmax)', fontweight='bold')
axes[0].set_ylabel('概率'); axes[0].tick_params(axis='x', rotation=45)

for ax, (name, probs) in zip(axes[1:], strategies.items()):
    ax.bar(labels, probs, color=colors, edgecolor='white')
    ax.set_title(name, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    max_idx = np.argmax(probs)
    ax.text(max_idx, probs[max_idx], f'{probs[max_idx]:.2f}',
            ha='center', va='bottom', fontsize=8, fontweight='bold')

fig.suptitle('同一组 logits 在不同采样策略下的概率分布', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

print("\n--- 各策略采样 1000 次的频率分布 ---")
for name, probs in strategies.items():
    if 'Greedy' in name:
        freq = strategies['Greedy']
    else:
        samples = sample_from_probs(probs, n_samples=1000)
        freq = np.bincount(samples, minlength=10) / 1000
    top3 = np.argsort(freq)[-3:][::-1]
    top3_str = ', '.join([f'{labels[i]}={freq[i]:.1%}' for i in top3])
    print(f"  {name:18s}: Top-3 -> {top3_str}")

---
## ✏️ 习题

> 在下方新建代码单元格作答。

1. （概念）离散随机变量和连续随机变量的本质区别是什么？掷骰子、身高测量、点击/不点击分别属于哪一类？
2. （概念）说出正态分布"68-95-99.7"规则的含义。为什么这个规则在异常检测中有用？
3. （概念）极大似然估计的核心直觉是什么？用一句话回答。
4. （代码）模拟两个公平骰子同时掷出，画出两骰子点数之和的概率分布（理论值 vs 模拟值）。10000 次模拟。
5. （代码）生成均值为 3、标准差为 1.5 的正态分布观测数据 100 个。用 MLE 估计 μ 和 σ，与真实值对比。重复 100 次实验，画出 μ̂ 的分布直方图（应当以真实 μ 为中心）。
6. （代码）给定 logits = [2.5, 1.8, 1.2, 0.8, 0.3, 0.1, 0.05, 0.02]，分别用 greedy、temperature=0.7、temperature=1.5、top-k=4、top-p=0.85 五种策略各采样 2000 次，用 2×3 子图画频率柱状图对比。

---

> 🔗 **章末钩子**：你已经能用概率分布描述数据了。但真实世界的数据不是孤立的——已知一个事件发生了，另一个事件的概率会怎么变？
> 预览：下一章——**条件概率与贝叶斯定理**，用已知信息更新判断。

> 💡 **提示**：完成后运行 `Kernel → Restart & Run All` 验证所有代码块。